# Calibrate A Day

This notebook performs calibration of a day's averaged spectrum. This is the same as `edges2k.c`.

In [ ]:
datadir: str = "/data7/smurray/edges/projects_with_nive/edges-bowman2018-pipeline/results/"
dayavgfile: str = "/data7/smurray/edges/projects_with_nive/edges-bowman2018-pipeline/output-notebooks/2016-250.averaged.gsh5"
calfile: str = "/data7/smurray/edges/projects_with_nive/edges-bowman2018-pipeline/output-notebooks/specal.txt"
ants11file: str = "/data7/smurray/edges/projects_with_nive/edges-bowman2018-pipeline/output-notebooks/2015_ants11_modelled.h5"
beamfactorfile: str = "/data7/smurray/edges/projects_with_nive/edges-bowman2018-pipeline/src/edges_pipelines/notebooks/beam_factor.hickle"

alan_output_dir = "/data4/smurray/edges/alans-pipeline/scripts/H2CaseFieldData/"

fmin: float = 50   # Minimum freq to keep in the data       
fmax: float = 100  # Maximum freq to keep in the data

cal_freq_min: float = 51  # Minimum freq to *use* (i.e. with weight>0)
cal_freq_max: float = 99

do_beam_correction: bool = True
inject_beam: bool = True
use_alans_beam_lsts: bool = True

## Imports and Data Load

In [ ]:
from pathlib import Path
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as un
import hickle

from pygsdata import GSData
from pygsdata.select import select_freqs

from edges_cal import Calibrator
from edges_cal.alanmode import read_specal_as_calibrator
from edges_cal import modelling as mdl

from edges_analysis.calibration.calibrate import apply_beam_correction, apply_loss_correction, apply_noise_wave_calibration
from edges_analysis.calibration.loss import low2_balun_connector_loss
from edges_analysis.averaging.freqbin import gauss_smooth
from edges_analysis.datamodel import add_model
from edges_analysis.filters import filters
from edges_pipelines import utils

In [ ]:
utils.print_versions()


In [ ]:
dayavgfile = Path(dayavgfile)

In [ ]:
year, day = dayavgfile.stem.split(".")[0].split("-")
year = int(year)
day = int(day)
datadir = Path(datadir)
alan_output_dir = Path(alan_output_dir) / str(utils.yday_to_alanday(year, day))

In [ ]:
pre_cal_data = GSData.from_file(dayavgfile)

In [ ]:
if calfile.endswith(".txt"):
    calobs = read_specal_as_calibrator(calfile, t_load=300, t_load_ns=1000)
else:
    calobs = Calibrator.from_calfile(calfile)

### Noise-Wave Calibration

In [ ]:
nw_data = select_freqs(pre_cal_data, freq_range=(fmin, fmax))

In [ ]:
nw_data = filters.flag_frequency_ranges(nw_data, freq_ranges=((cal_freq_min, cal_freq_max),), invert=True)

In [ ]:
alan_pre_cal = np.genfromtxt(alan_output_dir/"spectra_before_noise_waves.txt")

In [ ]:
nw_data = apply_noise_wave_calibration(
    nw_data,
    calobs = calobs,
    band='low',
    ant_s11_object=ants11file,
)

In [ ]:
alan_output_dir

In [ ]:
alan_spec_after_nw = np.genfromtxt(alan_output_dir/"spectra_after_noise_waves.txt", names=True)

In [ ]:
alan_ants11 = np.genfromtxt(alan_output_dir/"modeled_antenna_s11.txt")

In [ ]:
utils.plot_single_spectrum(nw_data, alan_spec_after_nw['spec'])

### Loss-Correction

In [ ]:
# Apply different loss corrections
loss_data = apply_loss_correction(
    nw_data, 
    ants11=ants11file, 
    ambient_temp=296*un.K, 
    loss_function=low2_balun_connector_loss, 
    use_approx_eps0=True
)
# no antenna correction
# no ground loss

In [ ]:
alan_spec_after_loss = np.genfromtxt(alan_output_dir / "spectra_after_loss.txt", names=True)

In [ ]:
utils.plot_single_spectrum(loss_data, alan_spec_after_loss['spec'])

### Beam Correction

In [ ]:
if do_beam_correction:
    if inject_beam:
        beamfac = np.genfromtxt(alan_output_dir/'beamcorr.txt')
        beam_data = loss_data.update(data=loss_data.data * beamfac[:, 3])
    else:
        beam = hickle.load(beamfactorfile)
        if use_alans_beam_lsts:
            lsts = []
            beamfacs = sorted(alan_output_dir.glob("beamfac*.txt"), key=lambda pth: int(pth.stem[7:]))
    
            for bf in beamfacs:
                with open(bf, 'r') as fl:
                    lsts.append(float(fl.readlines()[1].split("=")[-1].strip()))
            lsts = np.array(lsts)
        else:
            lsts = None
            
        beam_data = apply_beam_correction(
            loss_data, 
            beam=beam,
            freq_model = mdl.Fourier(
                n_terms=31, 
                transform=mdl.ShiftTransform(shift=75.0), 
                period=1.2*beam.nfreq * (beam.frequencies[1] - beam.frequencies[0])
            ),
            resample_beam_lsts = use_alans_beam_lsts,
            integrate_before_ratio = True,
            lsts=lsts,
            cut_to_data_lsts=not use_alans_beam_lsts,
        )
else:
    beam_data = loss_data

In [ ]:
# Get Alan's file after beam correction. NOTE: if beam correction is turned off in the script running the C-code, then this
# will NOT have beam correction actually applied, and will be the same as the previous plot.
alan_after_beam = np.genfromtxt(alan_output_dir/"spectra_after_beam.txt", names=True)

In [ ]:
utils.plot_single_spectrum(beam_data, alan_after_beam['spec'])

### Smooth Over Frequencies Again

In [ ]:
data = add_model(beam_data, model=mdl.PhysicalLin(n_terms=3, spectral_index=-2.5, f_center=75.0, with_cmb=False), nsamples_strategy='flagged-nsamples-uniform')

In [ ]:
alan_mdl = np.genfromtxt(alan_output_dir /"second_smooth_input.txt", names=True)

In [ ]:
utils.plot_single_spectrum(data, alan_mdl['model'], attribute='model')

In [ ]:
data = gauss_smooth(data, size=8, nsmooth=2, decimate_at=0, maintain_flags=9, use_residuals=True, use_nsamples=False)

In [ ]:
alan_specavg = np.genfromtxt(alan_output_dir / "specavg_cal.txt", usecols=(1,3,9))

In [ ]:
utils.plot_single_spectrum(data, alan_specavg[:, 1])

In [ ]:
data.write_gsh5(dayavgfile.parent / dayavgfile.name.replace(".averaged.", ".finalspec."));